## Importing and Cleaning our Data¶


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns   # This makes our visualization look nice.
import matplotlib.pyplot as plt # This is to actually make visualizations

df = pd.read_csv("/kaggle/input/fifa-data-for-eda-and-stats/fifa_eda_stats.csv")
sns.set()
df.sample(5)   # I'm starting to use sample more than head since I get a more comprehensive look at the data at a glance

### We should classify positions

In [ ]:
df.isna().sum(), df.duplicated().sum()

In [ ]:
def money_to_float(x):
    if isinstance(x, str):
        x = x.replace('€', '').replace(',', '')
        if 'M' in x:
            return float(x.replace('M', '')) * 1_000_000
        if 'K' in x:
            return float(x.replace('K', '')) * 1_000
        if x == '':
            return None
    return float(x)
    
df['Wage'] = df['Wage'].apply(money_to_float)
df['Value'] = df['Value'].apply(money_to_float)
df['Release Clause'] = df['Release Clause'].apply(money_to_float)
df['Preferred Foot'].value_counts(normalize=True)
df = df.dropna(subset=['Preferred Foot'])

In [ ]:
df['Preferred Foot'].value_counts(normalize=True).plot(kind="bar")

In [ ]:
from sklearn.impute import SimpleImputer
num_cols = df.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
df[num_cols] = imputer.fit_transform(df[num_cols])
cat_cols = df.select_dtypes(include=['object']).columns
imp_cat = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imp_cat.fit_transform(df[cat_cols])

In [ ]:
df.isna().sum()

In [ ]:
df["Preferred Foot"] = df["Preferred Foot"].map({"Left": 1, "Right": 0})

# --- 2. Drop irrelevant columns ---
drop_cols = [
    "ID", "Name", "Club", "Nationality",
    "Jersey Number"
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# --- 5. Separate features + target ---
X = df.drop("Preferred Foot", axis=1)
X = pd.get_dummies(X)
y = df["Preferred Foot"]

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size =.20, random_state = 42, stratify=y)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    class_weight={0: 1, 1: 2},  
    max_depth=5
)
model = rf.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
from sklearn.metrics import precision_score, accuracy_score

precision = precision_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
print(f"Precision: {precision}\nAccuracy: {accuracy}")

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("True")

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print(importances)

In [ ]:
import matplotlib.pyplot as plt

importances.head(20).plot(kind='bar', figsize=(10,5))
plt.title("Top 20 Feature Importances")
plt.show()
